# DeepSeek RAG 系统

基于 **DeepSeek** 大语言模型与 **FAISS** 向量数据库的检索增强生成（Retrieval-Augmented Generation）系统完整实现。

系统架构：文档分块 → 向量化 → FAISS 索引 → 相似度检索 → Prompt 构造 → DeepSeek 生成回答。

## 一、导入依赖库

In [25]:
import os               # 操作系统接口模块，提供文件路径拼接、环境变量读取等功能
import json             # JSON 数据序列化与反序列化模块，用于保存/读取对话历史
import requests         # HTTP 客户端库，用于向 DeepSeek REST API 发送 POST 请求
import numpy as np      # 数值计算库，主要用于嵌入向量的矩阵运算与 FAISS 数据准备
from typing import List, Dict, Any, Optional  # 类型注解工具，提升代码可读性与 IDE 类型推断
from dataclasses import dataclass             # 数据类装饰器，自动生成 __init__、__repr__ 等方法
import pickle           # Python 对象序列化模块，用于持久化存储文档对象列表
import re               # 正则表达式模块，用于按段落切割长文本
from pathlib import Path  # 面向对象的文件路径操作工具，比 os.path 更易读

# ---- 向量化模型：按优先级自动选择可用后端 ----
# 优先级：sentence-transformers > transformers > TF-IDF > 随机向量
try:
    from sentence_transformers import SentenceTransformer  # 高质量句向量库，基于预训练 Transformer
    USE_SENTENCE_TRANSFORMERS = True   # 标志位：bool 类型，True 表示 sentence-transformers 可用
except ImportError:
    USE_SENTENCE_TRANSFORMERS = False  # 标志位：bool 类型，False 表示需要降级到次选方案
    try:
        from transformers import AutoTokenizer, AutoModel  # HuggingFace 通用分词器与模型加载工具
        import torch                                        # PyTorch 深度学习框架，用于模型推理
        USE_TRANSFORMERS = True   # 标志位：bool 类型，True 表示 transformers 可用
    except ImportError:
        USE_TRANSFORMERS = False  # 标志位：bool 类型，False 表示进一步降级至 TF-IDF 方案

import faiss  # Facebook AI 相似度搜索库，提供高效的稠密向量近邻检索（支持 CPU/GPU 加速）

## 二、数据结构定义

### 2.1 Document 文档数据类

In [26]:
@dataclass
class Document:
    """
    表示知识库中一个文本片段（chunk）的数据容器。

    使用 Python dataclass 装饰器自动生成 __init__、__repr__、__eq__ 等方法，
    避免手写样板代码。

    Attributes
    ----------
    id : str
        文档唯一标识符，格式为 "<文件名>_<块序号>" 或 "text_<文本序号>_<块序号>"，
        用于在检索结果中追溯来源。
    content : str
        文档的原始文本内容（经过分块后的单个片段）。
    metadata : Dict[str, Any], optional
        附加元数据字典，通常包含：
        - source (str)：来源文件名
        - chunk_id (int)：块在源文件中的序号
        - file_path (str)：文件绝对路径
        默认为 None。
    embedding : np.ndarray, optional
        文档内容对应的稠密向量表示，
        shape 为 (embedding_dim,)，其中 embedding_dim 为所选模型的输出维度
        （例如 all-MiniLM-L6-v2 输出 384 维）。
        文档入库前该字段为 None，由 VectorStore.add_documents() 填充。
    """
    id: str                                  # 文档唯一 ID，str 类型
    content: str                             # 文本内容，str 类型
    metadata: Dict[str, Any] = None          # 元数据字典，Dict[str, Any] 类型，默认 None
    embedding: Optional[np.ndarray] = None   # 嵌入向量，shape: (embedding_dim,)，默认 None（未向量化）

## 三、核心模块实现

### 3.1 DeepSeekClient — DeepSeek API 客户端

In [27]:
class DeepSeekClient:
    """
    封装 DeepSeek OpenAI 兼容 REST API 的轻量级 HTTP 客户端。

    通过标准 HTTP POST 请求调用 /chat/completions 端点，
    支持多轮对话消息列表、模型选择、生成长度和采样温度配置。

    Parameters
    ----------
    api_key : str
        DeepSeek 平台颁发的 API 密钥，用于 Bearer Token 鉴权。
    base_url : str, optional
        API 服务根地址（不含末尾斜杠，不含 /chat/completions 端点路径），
        默认 "https://api.holysheep.ai/v1"（DeepSeek 中转站）。
        内部自动拼接 /chat/completions 端点，因此只需传入根地址，
        不要把完整端点路径放入 base_url，否则会造成路径重复。
        Chat Completions（/chat/completions）是 OpenAI 制定的行业标准接口，
        DeepSeek、Anthropic、Ollama、LM Studio 及各类中转站均兼容，
        可直接替换 base_url 和 api_key 切换服务商，无需修改其他代码。
        注意：OpenAI 新版 Responses 接口（/v1/responses）与此不兼容，
        该接口目前仅 OpenAI 官方支持，DeepSeek 等第三方及中转站不支持。
        常用 base_url 参考：
        - 中转站（当前）: https://api.holysheep.ai/v1
        - DeepSeek 官方 : https://api.deepseek.com
        - OpenAI 官方   : https://api.openai.com/v1
        - 本地 Ollama   : http://localhost:11434/v1
        - 本地 LM Studio: http://localhost:1234/v1
    """

    def __init__(self, api_key: str, base_url: str = "https://api.holysheep.ai/v1"):
        """
        初始化客户端，设置鉴权请求头。

        Parameters
        ----------
        api_key : str
            DeepSeek API 密钥，str 类型（中转站与官方使用相同格式的密钥）。
        base_url : str
            API 根 URL，str 类型，默认中转站 "https://api.holysheep.ai/v1"。
            末尾不加 /，不含 /chat/completions，否则拼接端点后路径会重复。
        """
        self.api_key = api_key    # 保存 API 密钥，str 类型，用于后续请求鉴权
        self.base_url = base_url  # 保存 API 根地址，str 类型，末尾不含 / 和端点路径
        self.headers = {          # HTTP 请求头字典，Dict[str, str] 类型
            "Authorization": f"Bearer {api_key}",  # Bearer Token 认证头，格式 "Bearer <key>"
            "Content-Type": "application/json"     # 声明请求体为 JSON 格式
        }

    def chat_completion(
        self,
        messages: List[Dict[str, str]],
        model: str = "deepseek-v4-pro",
        max_tokens: int = 2048,
        temperature: float = 0.7
    ) -> str:
        """
        调用 DeepSeek /chat/completions 接口，返回模型生成的文本。

        Parameters
        ----------
        messages : List[Dict[str, str]]
            对话消息列表，每个元素格式为 {"role": "user"/"assistant"/"system", "content": "..."}。
            列表长度决定上下文轮数（受模型 token 窗口限制）。
        model : str
            模型 ID，默认 "deepseek-v4-pro。
        max_tokens : int
            单次生成的最大 token 数，默认 2048，控制输出长度上限。
        temperature : float
            采样温度，范围 [0, 2]，值越低输出越确定性，默认 0.7 平衡多样性与准确性。

        Returns
        -------
        str
            模型生成的回复文本，str 类型；请求失败时返回空字符串 ""。
        """
        url = f"{self.base_url}/chat/completions"
        # 拼接完整端点 URL，str 类型
        # 示例：https://api.holysheep.ai/v1/chat/completions（中转站）
        # /chat/completions 是 OpenAI Chat Completions 标准接口，区别于 OpenAI 新版
        # /v1/responses（Responses 接口），后者仅 OpenAI 官方支持，中转站不支持

        payload = {                     # JSON 请求体，Dict[str, Any] 类型
            "model": model,             # 指定调用的模型 ID，str 类型
            "messages": messages,       # 传入多轮对话消息列表，List[Dict] 类型
            "max_tokens": max_tokens,   # 限制生成 token 数，int 类型
            "temperature": temperature, # 控制生成随机性，float 类型
            "stream": False             # 关闭流式返回，一次性获取完整响应
        }

        try:
            response = requests.post(url, headers=self.headers, json=payload)
            # requests.post(url, headers, json) 发送 HTTP POST 请求：
            #   url     : str，完整请求地址，即上方拼接好的端点 URL
            #   headers : Dict[str, str]，请求头，含 Authorization（鉴权）和 Content-Type
            #   json    : Dict，请求体，requests 自动将其序列化为 JSON 字符串并设置 Content-Type
            # 返回值 response: requests.Response 对象，主要属性：
            #   .status_code : int，HTTP 状态码（200 成功，401 未授权，429 超限，500 服务器错误）
            #   .headers     : Dict，响应头
            #   .text        : str，响应体原始文本
            #   .json()      : 将响应体解析为 Python 字典（需响应体为合法 JSON）
            response.raise_for_status()
            # 若 HTTP 状态码 >= 400（如 401 未授权、429 超限），抛出 HTTPError 异常

            result = response.json()
            # result: Dict[str, Any] 类型，OpenAI 兼容响应体，包含 choices、usage 等字段
            return result["choices"][0]["message"]["content"]
            # 逐层访问嵌套结构，四个 [] 依次取出：
            #   result                              → 完整响应字典 Dict
            #   result["choices"]                   → 候选回答列表 List（支持多候选，由参数 n 控制，默认 n=1）
            #   result["choices"][0]                → 第 0 个候选回答 Dict（含 message、finish_reason 等字段）
            #   result["choices"][0]["message"]     → 消息字典 Dict（含 role、content 字段）
            #   result["choices"][0]["message"]["content"] → 模型生成的回答文本 str，即最终返回值

        except requests.exceptions.RequestException as e:
            print(f"API 请求失败: {e}")   # 打印网络错误、超时、认证失败等请求级异常
            return ""                     # 返回空字符串，让调用方做降级处理

        except KeyError as e:
            print(f"响应字段缺失: {e}")   # 打印因响应结构不符预期导致的键错误
            return ""                     # 返回空字符串，防止程序崩溃

### 3.2 DocumentProcessor — 文档分块处理器

In [28]:
class DocumentProcessor:
    """
    负责将原始文本文件读取并切分为固定大小的文本块（chunk），
    为后续向量化和 FAISS 索引构建做数据准备。

    切分策略（两步）：
    第一步：用正则（匹配"换行 + 任意空白 + 换行"的连续空行）将全文切成若干段落；
    第二步：贪心合并——从第一个段落开始，不断将后续段落拼入当前块，
    只要累计字符数不超过 chunk_size 就继续拼，一旦超过则保存当前块、
    以该段落重新开始新块。切分点始终在段落边界，不在段落内部硬截断，
    因此单个段落超过 chunk_size 时会独立成块（块大小可能超限）。

    Parameters
    ----------
    chunk_size : int
        每个文本块的最大字符数，默认 512。
    chunk_overlap : int
        相邻文本块的重叠字符数（预留参数，当前版本未启用），默认 50。
    """

    def __init__(self, chunk_size: int = 512, chunk_overlap: int = 50):
        """
        初始化文档处理器，设置分块策略参数。

        Parameters
        ----------
        chunk_size : int
            文本块最大字符数，int 类型，默认 512。
        chunk_overlap : int
            块间重叠字符数，int 类型（预留），默认 50。
        """
        self.chunk_size = chunk_size       # 每块最大字符数，int 类型
        self.chunk_overlap = chunk_overlap  # 块间重叠字符数，int 类型（当前版本未启用）

    def load_text_file(self, file_path: str) -> str:
        """
        以 UTF-8 编码读取文本文件全部内容。

        Parameters
        ----------
        file_path : str
            待读取的文件绝对或相对路径，str 类型。

        Returns
        -------
        str
            文件的完整文本内容，str 类型，长度为文件字符总数。
        """
        with open(file_path, 'r', encoding='utf-8') as f:  # 以只读文本模式打开，UTF-8 编码防乱码
            return f.read()                                  # 返回文件全部内容，str 类型

    def split_text(self, text: str) -> List[str]:
        """
        将长文本按段落切分为不超过 chunk_size 字符的文本块列表。

        Parameters
        ----------
        text : str
            待切分的原始长文本，str 类型。

        Returns
        -------
        List[str]
            文本块列表，每个元素为 str 类型，首尾空白已去除。
            列表长度为实际分块数（单个超长段落的块字符数可能超过 chunk_size）。
        """
        paragraphs = re.split(r'\n\s*\n', text)
        # 使用正则按连续空行切分段落，返回 List[str]
        # 正则含义：\n 换行，\s* 任意空白字符（含空格），\n 再换行（即段落分隔的空行）

        chunks = []          # 已完成的文本块列表，List[str] 类型，初始为空
        current_chunk = ""   # 当前正在拼接的块，str 类型，初始为空字符串

        for paragraph in paragraphs:              # 逐段落迭代，paragraph: str 类型
            if len(current_chunk) + len(paragraph) > self.chunk_size:
                # 加入该段落后总字符数超过 chunk_size：先保存当前块，再开新块
                if current_chunk:                    # 当前块非空才保存，避免添加空字符串
                    chunks.append(current_chunk.strip())  # strip() 去除首尾空白后加入结果
                current_chunk = paragraph            # 以当前段落单独开启新块
            else:
                # 未超限：将段落拼接到当前块，首个段落直接赋值，后续段落用空行连接
                current_chunk += "\n\n" + paragraph if current_chunk else paragraph

        if current_chunk:                         # 循环结束后当前块仍有内容（最后一块）
            chunks.append(current_chunk.strip())  # 保存最后一块

        return chunks  # 返回 List[str]，长度为最终分块总数

    def process_file(self, file_path: str) -> List[Document]:
        """
        读取指定文件并将其内容转换为 Document 对象列表。

        每个文本块生成一个 Document，metadata 记录来源文件名、块序号和文件路径。

        Parameters
        ----------
        file_path : str
            待处理的文件路径，str 类型（绝对或相对均可）。

        Returns
        -------
        List[Document]
            Document 对象列表，长度等于文本块数量。
            每个 Document 的 embedding 字段为 None（由 VectorStore 后续填充）。
        """
        text = self.load_text_file(file_path)   # 读取文件全文，str 类型
        chunks = self.split_text(text)           # 分块结果，List[str] 类型，长度为块数

        documents = []                            # Document 对象列表，List[Document] 类型
        filename = Path(file_path).name           # 提取不含目录的文件名，str 类型，如 "data.txt"

        for i, chunk in enumerate(chunks):        # 遍历每个文本块，i: int 块序号，chunk: str 块内容
            doc = Document(
                id=f"{filename}_{i}",             # ID 格式："<文件名>_<块序号>"，str 类型
                content=chunk,                    # 块文本内容，str 类型
                metadata={                        # 元数据字典，Dict[str, Any] 类型
                    "source": filename,           # 来源文件名，str 类型
                    "chunk_id": i,                # 块序号，int 类型，从 0 开始计
                    "file_path": file_path        # 文件完整路径，str 类型
                }
            )
            documents.append(doc)                 # 追加到结果列表

        return documents  # 返回 List[Document]，长度等于分块总数

### 3.3 EmbeddingModel — 嵌入模型包装器

In [29]:
class EmbeddingModel:
    """
    嵌入模型的统一封装类，按优先级自动选择可用的向量化后端：

    优先级（由高到低）：
    1. sentence-transformers：语义质量最高，推荐使用
    2. HuggingFace transformers + PyTorch：次选，需要额外安装 torch
    3. scikit-learn TF-IDF：降级方案，语义质量较低但无深度学习依赖
    4. 随机向量：最终兜底，仅用于调试，无任何语义含义

    关于 sentence-transformers 与 transformers 的关系：
    sentence-transformers 底层依赖 transformers，并非完全独立，
    但它在 transformers 之上做了专门封装，解决了原生 transformers
    生成句向量时需要手动处理的诸多细节：
    - 原生 transformers：需手动分词、推理、选择池化策略（均值/CLS）、归一化
    - sentence-transformers：model.encode(text) 一行完成上述全部步骤
    此外，sentence-transformers 提供的模型权重经过句对相似度任务专门微调，
    语义检索效果显著优于直接使用 BERT 等基础模型的原始权重。
    代码中将两者并列是降级方案：优先用 sentence-transformers，
    安装失败时降级用原生 transformers 手动复现相同功能。

    Parameters
    ----------
    model_name : str
        sentence-transformers 模型名称，默认 "all-MiniLM-L6-v2"（输出 384 维，速度快、效果好）。

        可用模型均托管于 HuggingFace Hub（https://huggingface.co/sentence-transformers），
        传入模型名后首次运行自动下载并缓存到本地。

        常用英文模型：
        - all-MiniLM-L6-v2      : 384 维，速度极快，综合性能佳，最常用（默认）
        - all-MiniLM-L12-v2     : 384 维，比 L6 精度略高，速度略慢
        - all-mpnet-base-v2     : 768 维，精度最高的英文通用模型，速度较慢
        - paraphrase-MiniLM-L6-v2: 384 维，专为语义相似度/复述任务优化

        常用中文 / 多语言模型：
        - paraphrase-multilingual-MiniLM-L12-v2: 384 维，支持 50+ 语言含中文，速度快
        - paraphrase-multilingual-mpnet-base-v2 : 768 维，多语言高精度，速度较慢
        - shibing624/text2vec-base-chinese       : 768 维，专为中文优化，中文场景推荐

        选择建议：
        - 纯英文内容         → all-MiniLM-L6-v2（默认，速度与效果平衡）
        - 中文 / 中英混合    → paraphrase-multilingual-MiniLM-L12-v2 或 text2vec-base-chinese
        - 追求最高精度       → all-mpnet-base-v2（英文）
        - 资源受限/快速原型  → all-MiniLM-L6-v2
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        初始化嵌入模型，并触发自动后端选择逻辑。

        Parameters
        ----------
        model_name : str
            向量化模型名称，str 类型，默认 "all-MiniLM-L6-v2"。
            首次使用时自动从 HuggingFace Hub 下载模型并缓存到本地，
            后续运行直接加载本地缓存，无需重复下载。
            可选模型及选择建议详见类 docstring。
        """
        self.model_name = model_name  # 保存模型名称，str 类型，用于 sentence-transformers 后端
        self.model = None             # 模型对象，初始为 None，由 _initialize_model 赋值
        self.tokenizer = None         # 分词器对象，初始为 None，仅 transformers 后端使用
        self._initialize_model()      # 按优先级尝试初始化，选择最优可用后端

    def _initialize_model(self):
        """
        按优先级自动选择并初始化嵌入后端。
        调用链：sentence_transformers → _fallback_to_transformers → _fallback_to_simple。
        """
        if USE_SENTENCE_TRANSFORMERS:   # 优先使用 sentence-transformers（质量最高）
            try:
                self.model = SentenceTransformer(self.model_name,cache_folder="./model_cache")
                # self.model: SentenceTransformer 类型，封装预训练 Transformer 编码器
                self.model_type = "sentence_transformers"  # 后端标识，str 类型
                print(f"使用 sentence-transformers 模型: {self.model_name}")
            except Exception as e:
                print(f"sentence-transformers 初始化失败: {e}")
                self._fallback_to_transformers()           # 初始化失败则降级
        else:
            self._fallback_to_transformers()               # 库不可用则直接降级

    def _fallback_to_transformers(self):
        """
        尝试使用 HuggingFace transformers 作为嵌入后端。
        失败时进一步降级到 TF-IDF。
        """
        if USE_TRANSFORMERS:   # transformers 库可用时进入
            try:
                model_name = "sentence-transformers/all-MiniLM-L6-v2"  # HuggingFace Hub 模型 ID
                self.tokenizer = AutoTokenizer.from_pretrained(model_name)
                # self.tokenizer: AutoTokenizer 类型，负责文本 → token ID 转换
                self.model = AutoModel.from_pretrained(model_name)
                # self.model: AutoModel 类型，加载预训练权重，含 Transformer 编码器结构
                self.model_type = "transformers"  # 后端标识，str 类型
                print(f"使用 transformers 模型: {model_name}")
            except Exception as e:
                print(f"transformers 初始化失败: {e}")
                self._fallback_to_simple()   # 进一步降级
        else:
            self._fallback_to_simple()       # 库不可用则直接降级

    def _fallback_to_simple(self):
        """
        使用 scikit-learn TF-IDF 作为最低可用嵌入后端。
        若 sklearn 也不可用，则退化为随机向量（仅调试用）。

        TF-IDF（词频-逆文档频率）将文本转为稀疏词袋向量，
        设置 max_features=384 使输出维度与 sentence-transformers 对齐，便于复用 FAISS 索引。
        """
        print("降级至简单 TF-IDF 嵌入方案")
        self.model_type = "tfidf"    # 后端标识，str 类型
        try:
            from sklearn.feature_extraction.text import TfidfVectorizer
            # TfidfVectorizer：稀疏 TF-IDF 向量化器，输出稀疏矩阵
            self.model = TfidfVectorizer(max_features=384, stop_words='english')
            # max_features=384：限制词表大小（即向量维度）与 sentence-transformers 对齐
            # stop_words='english'：去除英文停用词，降低噪声
            self.fitted = False      # 标志位，bool 类型，False 表示词表尚未在语料上拟合
        except ImportError:
            print("警告: sklearn 不可用，将使用随机向量（结果无语义含义）")
            self.model_type = "random"  # 最终降级标识，str 类型

    def encode(self, texts: List[str], show_progress_bar: bool = True) -> np.ndarray:
        """
        将文本列表编码为稠密嵌入矩阵，根据 model_type 路由到对应后端实现。

        Parameters
        ----------
        texts : List[str]
            待编码的文本列表，长度为 N（文档数或单条查询时为 1）。
        show_progress_bar : bool
            是否显示编码进度条，仅 sentence_transformers 后端有效，默认 True。

        Returns
        -------
        np.ndarray
            嵌入矩阵，shape: (N, embedding_dim)，
            其中 N 为文本数量，embedding_dim 为向量维度（通常为 384）。
        """
        if self.model_type == "sentence_transformers":
            return self.model.encode(texts, show_progress_bar=show_progress_bar)
            # 返回 np.ndarray，shape: (N, embedding_dim)，N 为文本数，embedding_dim 为模型输出维度

        elif self.model_type == "transformers":
            return self._encode_with_transformers(texts)   # 使用 transformers 后端逐条编码

        elif self.model_type == "tfidf":
            return self._encode_with_tfidf(texts)          # 使用 TF-IDF 后端批量编码

        else:
            return self._encode_random(texts)              # 随机向量（调试兜底）

    def _encode_with_transformers(self, texts: List[str]) -> np.ndarray:
        """
        使用 HuggingFace transformers 对文本逐条编码（分词 → 推理 → 均值池化）。

        Parameters
        ----------
        texts : List[str]
            待编码文本列表，长度为 N。

        Returns
        -------
        np.ndarray
            嵌入矩阵，shape: (N, hidden_size)，
            N 为文本数量，hidden_size 为模型隐藏层维度（all-MiniLM-L6-v2 为 384）。
        """
        embeddings = []  # 中间结果列表，List[np.ndarray]，每个元素 shape: (hidden_size,)

        for text in texts:  # 逐条处理，避免批量 padding 导致内存浪费
            inputs = self.tokenizer(
                text,
                return_tensors="pt",   # 返回 PyTorch Tensor 格式，键含 input_ids、attention_mask
                truncation=True,       # 超过 max_length 时截断，防止超出位置编码范围
                padding=True,          # 对批内序列做 padding（此处批大小为 1）
                max_length=512         # 最大 token 数，all-MiniLM-L6-v2 支持上限为 512
            )
            # inputs: Dict[str, torch.Tensor]，input_ids shape: (1, seq_len)

            with torch.no_grad():   # 关闭梯度计算，减少显存占用，加速推理
                outputs = self.model(**inputs)
                # outputs.last_hidden_state: torch.Tensor，shape: (1, seq_len, hidden_size)
                # 1: batch_size；seq_len: 序列长度（含 [CLS]、[SEP] 等特殊 token）；hidden_size: 隐藏层维度

                embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
                # mean(dim=1)：对 seq_len 维度取均值（均值池化），得 shape: (1, hidden_size)
                # squeeze()：去掉 batch 维度，得 shape: (hidden_size,)
                # .numpy()：转为 np.ndarray，shape: (hidden_size,)
                embeddings.append(embedding)

        return np.array(embeddings)
        # 将 List[np.ndarray] 堆叠为矩阵，返回 np.ndarray，shape: (N, hidden_size)

    def _encode_with_tfidf(self, texts: List[str]) -> np.ndarray:
        """
        使用 TF-IDF 对文本列表编码为稠密矩阵。

        首次调用时在传入语料上拟合词表与 idf 权重；后续调用直接转换。

        Parameters
        ----------
        texts : List[str]
            待编码文本列表，长度为 N。

        Returns
        -------
        np.ndarray
            TF-IDF 特征矩阵，shape: (N, max_features)，
            N 为文本数量，max_features 为词表大小（此处限定为 384）。
        """
        if not self.fitted:        # 首次调用时拟合词表（统计词频、计算 idf 权重）
            self.model.fit(texts)  # 在传入语料上拟合，构建词表和 idf 向量
            self.fitted = True     # 置为 True，后续调用跳过拟合步骤

        return self.model.transform(texts).toarray()
        # transform 返回稀疏矩阵（scipy.sparse.csr_matrix）
        # .toarray() 转为 np.ndarray，shape: (N, 384)

    def _encode_random(self, texts: List[str]) -> np.ndarray:
        """
        生成随机嵌入向量，仅用于缺少深度学习框架时的占位调试。

        警告：随机向量不携带任何语义信息，检索结果完全随机且无意义。

        Parameters
        ----------
        texts : List[str]
            待编码文本列表，长度为 N（仅用于确定输出行数）。

        Returns
        -------
        np.ndarray
            标准正态随机矩阵，shape: (N, 384)，
            N 为文本数量，384 为固定占位维度（与其他后端保持一致）。
        """
        print("警告: 当前使用随机嵌入，检索结果无语义意义，请安装 sentence-transformers")
        return np.random.randn(len(texts), 384)
        # np.random.randn 生成标准正态分布随机数，shape: (N, 384)
        # N = len(texts)：文本数量；384：占位向量维度

### 3.4 VectorStore — 向量索引与检索

In [30]:
class VectorStore:
    """
    基于 FAISS IndexFlatIP（内积相似度）的向量存储与近邻检索模块。

    主要职责：
    1. 调用 EmbeddingModel 将文档文本批量向量化；
    2. 构建 FAISS 索引支持毫秒级近邻搜索；
    3. 提供持久化接口（pickle 存文档元数据 + faiss 二进制存索引）。

    相似度度量：先对所有向量做 L2 归一化，再用内积（IndexFlatIP），
    等价于余弦相似度，确保不同长度文本的相似度比较一致。

    Parameters
    ----------
    embedding_model_name : str
        传递给 EmbeddingModel 的模型名称，默认 "all-MiniLM-L6-v2"。
    """

    def __init__(self, embedding_model_name: str = "all-MiniLM-L6-v2"):
        """
        初始化向量存储，创建嵌入模型并清空文档列表和索引。

        Parameters
        ----------
        embedding_model_name : str
            向量化模型名称，str 类型，传递给 EmbeddingModel。
        """
        self.embedding_model = EmbeddingModel(embedding_model_name)
        # self.embedding_model: EmbeddingModel 类型，负责文本 → 向量的转换
        self.documents: List[Document] = []  # 文档对象列表，List[Document] 类型，与 FAISS 索引行一一对应
        self.index = None                    # FAISS 索引，faiss.IndexFlatIP 类型，未初始化为 None
        self.dimension = None                # 向量维度，int 类型，由首次 add_documents 时确定

    def add_documents(self, documents: List[Document]):
        """
        向量化文档列表并将其加入 FAISS 索引。

        执行步骤：
        1. 批量调用 embedding_model.encode() 生成嵌入矩阵；
        2. 将向量回写到各 Document.embedding 字段；
        3. 追加到内部文档列表；
        4. 重建 FAISS 索引（全量重建保证索引与文档列表同步）。

        Parameters
        ----------
        documents : List[Document]
            待入库的 Document 对象列表，长度为 M（新增文档数）。
            函数执行后，每个 Document.embedding 字段将被填充为 shape: (embedding_dim,) 的向量。
        """
        existing_ids = {doc.id for doc in self.documents}
        # 已入库文档的 ID 集合，Set[str] 类型，用于去重判断
        # 使用集合（set）而非列表，查找复杂度 O(1)

        documents = [doc for doc in documents if doc.id not in existing_ids]
        # 过滤掉已存在 ID 的文档，避免重复运行 cell 时重复入库
        # Jupyter Notebook 中重复执行 add_documents 会导致同一文档多次追加，
        # 造成搜索结果出现重复项，此处按 ID 去重后只处理真正新增的文档

        if not documents:   # 全部文档已存在，无需重复入库
            print("所有文档已存在于知识库中，跳过入库")
            return

        print(f"正在向量化 {len(documents)} 个文档...")

        contents = [doc.content for doc in documents]
        # 提取所有文档文本，List[str] 类型，长度 M（去重后的新增文档数）

        embeddings = self.embedding_model.encode(contents, show_progress_bar=True)
        # embeddings: np.ndarray，shape: (M, embedding_dim)
        # M: 新增文档数量；embedding_dim: 向量维度（如 all-MiniLM-L6-v2 输出 384 维）

        for doc, embedding in zip(documents, embeddings):  # 将向量逐一回写到文档对象的 embedding 字段
            doc.embedding = embedding  # embedding shape: (embedding_dim,)，np.ndarray 类型

        self.documents.extend(documents)  # 将去重后的新文档追加到总文档列表，List[Document] 类型

        self._build_index()  # 重建 FAISS 索引，保证索引与 self.documents 完全同步

        print(f"成功入库 {len(documents)} 个文档，当前知识库总量: {len(self.documents)} 个")

    def _build_index(self):
        """
        基于当前所有文档的嵌入向量重建 FAISS IndexFlatIP 索引。

        采用 L2 归一化 + 内积（IndexFlatIP）等价于余弦相似度精确搜索，
        适合数据量在数万以内的场景（精确搜索，无近似误差）。

        FAISS 常见索引类型对比（可按规模替换，接口一致）：
        索引类型          搜索方式                精度      速度    适用规模
        IndexFlatIP      暴力内积（当前使用）     100% 精确  慢      万级以下
        IndexFlatL2      暴力 L2 距离            100% 精确  慢      万级以下
        IndexIVFFlat     先 K-Means 聚类再搜索   近似       快      十万~百万级
        IndexHNSW        多层图结构导航搜索       近似       很快    百万级以上
        IndexIVFPQ       聚类 + 向量量化压缩      近似       最快    亿级
        """
        if not self.documents:   # 文档列表为空时直接返回，避免构建空索引
            return

        embeddings = np.array([doc.embedding for doc in self.documents])
        # 将所有文档嵌入向量堆叠为矩阵，np.ndarray
        # shape: (total_docs, embedding_dim)
        # total_docs: 知识库中当前文档总数；embedding_dim: 向量维度（如 384）

        self.dimension = embeddings.shape[1]  # 记录向量维度，int 类型，索引维度参数

        self.index = faiss.IndexFlatIP(self.dimension)
        # 创建空的内积索引容器，faiss.IndexFlatIP 类型，此时容器内无任何向量
        # 命名拆解：Index（索引容器）+ Flat（暴力搜索，不加速）+ IP（Inner Product 内积）
        # Flat 意味着 search() 时仍与每个向量逐一计算内积，时间复杂度 O(N)，无近似加速
        # "IP"只声明了相似度度量方式（内积），此处不做任何内积计算
        # 整个 _build_index 阶段（创建容器→归一化→add）均不计算内积，只是"建库存数据"
        # 内积计算仅在 search() 被调用时发生：查询向量与库中所有向量逐一做内积后排序
        # RAG 知识库通常只有数百~数千个文档块，暴力搜索耗时仅几毫秒，无需加速索引
        # 若规模扩大到十万级以上，可替换为 IndexIVFFlat（聚类跳过大量向量）
        # 或 IndexHNSW（图结构导航跳跃搜索），接口相同，只需改这一行

        faiss.normalize_L2(embeddings)
        # 原地 L2 归一化：每行向量除以其 L2 范数
        # 归一化后 shape 不变：(total_docs, embedding_dim)
        # 目的：使内积 == 余弦相似度，统一相似度度量标准

        self.index.add(embeddings.astype(np.float32))
        # 将归一化后的 float32 矩阵存入索引容器，FAISS 要求输入为 float32 类型
        # 此步仍不计算任何内积，只是把向量逐条写入索引的内部存储
        # 内积计算发生在 search() 调用时：查询向量与库中所有向量逐一计算内积后排序
        # 添加后 self.index.ntotal == len(self.documents)，可用于验证入库数量

    def search(self, query: str, top_k: int = 5) -> List[Document]:
        """
        对查询文本执行近邻搜索，返回最相关的文档列表。

        Parameters
        ----------
        query : str
            用户查询文本，str 类型。
        top_k : int
            返回的最相关文档数量，默认 5。

        Returns
        -------
        List[Document]
            按余弦相似度降序排列的 Document 对象列表，长度 <= top_k。
            若索引为空（未添加文档）则返回 []。
        """
        if not self.index:   # 索引未建立（尚未调用 add_documents）时返回空列表
            return []

        query_embedding = self.embedding_model.encode([query])
        # 编码单条查询文本，np.ndarray，shape: (1, embedding_dim)
        # 1: batch_size（单条查询）；embedding_dim: 向量维度

        faiss.normalize_L2(query_embedding)
        # 对查询向量做 L2 归一化，保持与索引向量的归一化一致性
        # shape 不变：(1, embedding_dim)

        scores, indices = self.index.search(query_embedding.astype(np.float32), top_k)
        # search 返回两个 np.ndarray：
        # scores: shape (1, top_k)，每列为归一化内积分数（余弦相似度），值域 [-1, 1]，越大越相似
        # indices: shape (1, top_k)，每列为对应文档在 self.documents 中的下标，-1 表示填充位

        results = []  # 检索结果列表，List[Document] 类型
        for score, idx in zip(scores[0], indices[0]):    # 遍历第 0 条查询的 top_k 个结果
            if 0 <= idx < len(self.documents):            # 过滤无效下标：必须同时满足 >= 0 且 < 文档总数
                # FAISS 文档数不足 top_k 时用 -1 填充剩余位置
                # Python 中 -1 < len(...) 为 True，self.documents[-1] 会返回最后一个文档造成重复
                # 因此必须加 >= 0 判断才能正确过滤掉 FAISS 的 -1 填充值
                results.append(self.documents[idx])       # 根据下标取出对应 Document 对象

        return results  # 返回 List[Document]，长度 <= top_k，按相似度降序

    def save(self, file_path: str):
        """
        将文档列表与向量维度序列化为 pickle 文件，并将 FAISS 索引保存为二进制文件。

        Parameters
        ----------
        file_path : str
            基础保存路径，str 类型。
            实际生成文件：<file_path>（pickle）和 <file_path>.faiss（FAISS 索引）。
        """
        data = {                           # 需持久化的元数据，Dict[str, Any] 类型
            'documents': self.documents,   # 文档对象列表（含 embedding），List[Document]
            'dimension': self.dimension    # 向量维度，int 类型
        }
        with open(file_path, 'wb') as f:   # 以二进制写模式打开
            pickle.dump(data, f)           # 序列化为 pickle 格式写入文件

        if self.index:                                            # 仅在索引已建立时保存
            faiss.write_index(self.index, file_path + ".faiss")  # 将 FAISS 索引写入 .faiss 文件

    def load(self, file_path: str):
        """
        从磁盘恢复向量存储的完整状态（文档列表 + FAISS 索引）。

        Parameters
        ----------
        file_path : str
            基础文件路径，str 类型，与 save() 时保持一致。
        """
        with open(file_path, 'rb') as f:   # 以二进制读模式打开 pickle 文件
            data = pickle.load(f)           # 反序列化，data: Dict[str, Any] 类型

        self.documents = data['documents']  # 恢复文档列表，List[Document] 类型
        self.dimension = data['dimension']  # 恢复向量维度，int 类型

        faiss_path = file_path + ".faiss"
        if os.path.exists(faiss_path):                       # .faiss 文件存在时才加载
            self.index = faiss.read_index(faiss_path)        # 从文件加载 FAISS 索引，faiss.IndexFlatIP 类型

### 3.5 RAGSystem — RAG 系统主类

In [31]:
class RAGSystem:
    """
    检索增强生成（RAG）系统的顶层协调类，整合三大核心组件协同工作。

    完整工作流程：
    1. 用户输入问题 → VectorStore 检索最相关的 top_k 个文档块（向量相似度）；
    2. 将检索到的文档块按格式拼接为上下文字符串；
    3. 构造 prompt（系统指令 + 上下文 + 用户问题）发送给 DeepSeek；
    4. 返回包含回答、来源文档、上下文的结构化结果字典。

    可选多轮对话支持：将历史消息追加到 messages 列表中，
    最多保留最近 10 条（即最近 5 轮问答），控制 token 消耗。
    每轮对话追加 2 条消息（user 1条 + assistant 1条），10 条 ÷ 2 = 5 轮。

    Parameters
    ----------
    deepseek_api_key : str
        DeepSeek API 密钥，str 类型。
    embedding_model : str
        向量化模型名称，str 类型，默认 "all-MiniLM-L6-v2"。
    """

    def __init__(self, deepseek_api_key: str, embedding_model: str = "all-MiniLM-L6-v2"):
        """
        初始化 RAG 系统，组装并连接三大核心组件。

        Parameters
        ----------
        deepseek_api_key : str
            DeepSeek API 密钥，str 类型。
        embedding_model : str
            嵌入模型名称，str 类型，默认 "all-MiniLM-L6-v2"。
        """
        self.deepseek_client = DeepSeekClient(deepseek_api_key)
        # DeepSeek HTTP 客户端，DeepSeekClient 类型，负责调用 LLM 生成回答

        self.document_processor = DocumentProcessor()
        # 文档分块处理器，DocumentProcessor 类型，负责读文件并切分文本块

        self.vector_store = VectorStore(embedding_model)
        # 向量存储与检索模块，VectorStore 类型，负责向量化文档和相似度搜索

        self.conversation_history: List[Dict[str, str]] = []
        # 多轮对话历史，List[Dict[str, str]] 类型
        # 每个元素格式：{"role": "user"/"assistant", "content": "..."}

    def add_documents_from_files(self, file_paths: List[str]):
        """
        批量读取文件，分块后入库到向量存储。

        Parameters
        ----------
        file_paths : List[str]
            文件路径列表，List[str] 类型，每个元素为绝对或相对路径。
        """
        all_documents = []  # 汇总所有文件分块后的 Document 列表，List[Document] 类型

        for file_path in file_paths:                                       # 逐文件处理
            print(f"正在处理文件: {file_path}")
            documents = self.document_processor.process_file(file_path)    # 返回 List[Document]
            all_documents.extend(documents)                                 # 合并到总列表

        self.vector_store.add_documents(all_documents)                      # 批量向量化并入库

    def add_documents_from_text(self, texts: List[str], metadata_list: List[Dict] = None):
        """
        将原始文本列表分块后入库到向量存储。

        Parameters
        ----------
        texts : List[str]
            原始文本列表，List[str] 类型，长度为文本总数 T。
        metadata_list : List[Dict], optional
            与 texts 等长的元数据列表，List[Dict] 类型；
            为 None 时每个文档的 metadata 默认为 {"source": "text_<i>"}。
        """
        documents = []  # 分块后的 Document 列表，List[Document] 类型

        for i, text in enumerate(texts):                       # 遍历每条原始文本，i: int 文本序号
            chunks = self.document_processor.split_text(text)  # 分块，List[str] 类型

            for j, chunk in enumerate(chunks):                 # 遍历每个文本块，j: int 块序号
                doc = Document(
                    id=f"text_{i}_{j}",                        # ID 格式："text_<文本序号>_<块序号>"，str 类型
                    content=chunk,                             # 块文本内容，str 类型
                    metadata=metadata_list[i] if metadata_list else {"source": f"text_{i}"}
                    # 有元数据列表时取对应项（Dict 类型），否则生成默认元数据
                )
                documents.append(doc)

        self.vector_store.add_documents(documents)              # 批量向量化并入库

    def _build_context(self, relevant_docs: List[Document]) -> str:
        """
        将检索到的相关文档列表拼接为格式化的上下文字符串。

        Parameters
        ----------
        relevant_docs : List[Document]
            检索结果文档列表，List[Document] 类型，长度 <= top_k。

        Returns
        -------
        str
            格式化上下文字符串，每个文档以 "文档 N:" 为标题，文档间以空行分隔。
            整体作为 prompt 的上下文部分传给 DeepSeek。
        """
        context_parts = []  # 上下文片段列表，List[str] 类型，最终 join 为单一字符串

        for i, doc in enumerate(relevant_docs):        # 遍历每个相关文档，i: int 从 0 开始
            context_parts.append(f"文档 {i + 1}:")    # 添加文档序号标题（从 1 开始展示）
            context_parts.append(doc.content)          # 添加文档文本内容，str 类型
            context_parts.append("")                   # 添加空字符串，join 后形成空行分隔

        return "\n".join(context_parts)  # 用换行符拼接所有片段，返回 str 类型的上下文字符串

    def _build_prompt(self, query: str, context: str) -> str:
        """
        构造发送给 DeepSeek 的完整提示词（prompt）。

        提示词结构：角色设定 → 上下文信息 → 用户问题 → 回答引导词。
        使用 f-string 模板将上下文和问题嵌入固定框架，引导模型基于上下文回答。

        Parameters
        ----------
        query : str
            用户原始问题，str 类型。
        context : str
            由 _build_context() 生成的上下文字符串，str 类型。

        Returns
        -------
        str
            完整 prompt 字符串，str 类型，直接传入 messages 列表的 content 字段。
        """
        prompt = f"""你是一个专业的 AI 助手。请严格根据以下上下文信息回答用户问题。

上下文信息：
{context}

用户问题：{query}

请基于上下文给出准确、详细的回答。若上下文信息不足以回答问题，请明确说明。

回答："""
        return prompt  # 返回完整 prompt 字符串，str 类型

    def query(
        self,
        question: str,
        top_k: int = 5,
        use_history: bool = False
    ) -> Dict[str, Any]:
        """
        执行一次完整的 RAG 查询：检索 → 构造 prompt → 调用 LLM → 返回结构化结果。

        Parameters
        ----------
        question : str
            用户问题，str 类型。
        top_k : int
            检索的相关文档数量，默认 5。
        use_history : bool
            是否启用多轮对话历史，默认 False。
            启用时将历史消息前置到 messages 列表，并在回答后更新历史。

        Returns
        -------
        Dict[str, Any]
            结果字典，包含以下键值：
            - "answer" : str，LLM 生成的回答文本
            - "sources" : List[Dict]，来源信息列表，每项含：
                - "document_id" (str)：文档唯一 ID
                - "content_preview" (str)：内容前 200 字预览
                - "metadata" (Dict)：文档元数据
            - "context" : str，传给 LLM 的完整上下文字符串
            - "relevant_documents" : int，实际检索到的相关文档数量
        """
        relevant_docs = self.vector_store.search(question, top_k)
        # 调用 FAISS 检索，返回 List[Document]，按余弦相似度降序排列，长度 <= top_k

        if not relevant_docs:    # 未检索到任何文档时（知识库为空或查询偏差过大）
            return {
                "answer": "抱歉，知识库中暂无与该问题相关的文档。",
                "sources": [],
                "context": ""
            }

        context = self._build_context(relevant_docs)  # 构造上下文字符串，str 类型

        messages: List[Dict[str, str]] = []  # 消息列表，List[Dict[str, str]] 类型，传给 DeepSeek API

        if use_history:                                    # 多轮模式：历史消息前置
            messages.extend(self.conversation_history)    # 将已有历史追加到列表头部

        prompt = self._build_prompt(question, context)    # 构造当前轮完整 prompt，str 类型
        messages.append({"role": "user", "content": prompt})
        # 将用户消息追加到列表末尾，格式：{"role": "user", "content": prompt}

        answer = self.deepseek_client.chat_completion(messages)
        # 调用 DeepSeek API 获取回答，返回 str 类型（失败时返回 ""）

        if use_history:   # 多轮模式：将本轮问答追加到历史（存储原始问题而非 prompt，减少冗余）
            self.conversation_history.append({"role": "user", "content": question})
            # 存储用户原始问题，str 类型，不含 RAG 上下文
            self.conversation_history.append({"role": "assistant", "content": answer})
            # 存储模型回答，str 类型
            # 每轮对话追加 2 条消息（user + assistant），N 轮 = 2N 条

            if len(self.conversation_history) > 10:              # 历史超过 10 条时裁剪
                self.conversation_history = self.conversation_history[-10:]
                # 保留最近 10 条 = 最近 5 轮对话，List[Dict] 类型，防止 token 消耗过大
                # 原理：每轮 = user 1条 + assistant 1条 = 2条，10条 ÷ 2 = 5轮

        sources = []  # 来源信息列表，List[Dict[str, Any]] 类型
        for doc in relevant_docs:    # 遍历检索到的相关文档，构造来源摘要
            source_info = {
                "document_id": doc.id,    # 文档唯一 ID，str 类型
                "content_preview": (      # 内容预览，str 类型：超 200 字截断并加省略号，否则完整显示
                    doc.content[:200] + "..." if len(doc.content) > 200 else doc.content
                ),
                "metadata": doc.metadata  # 元数据字典，Dict[str, Any] 类型
            }
            sources.append(source_info)   # 追加到来源列表

        return {                                           # 返回结构化结果字典，Dict[str, Any] 类型
            "answer": answer,                             # LLM 生成的回答，str 类型
            "sources": sources,                           # 来源文档信息列表，List[Dict] 类型
            "context": context,                           # 传给 LLM 的上下文字符串，str 类型
            "relevant_documents": len(relevant_docs)      # 实际相关文档数量，int 类型
        }

    def clear_history(self):
        """清空多轮对话历史，将 conversation_history 重置为空列表。"""
        self.conversation_history = []  # 重置为空 List，List[Dict[str, str]] 类型

    def save_system(self, file_path: str):
        """
        持久化保存 RAG 系统状态，包括向量库和对话历史。

        生成文件：
        - <file_path>        ：pickle 格式，存储文档列表与向量维度
        - <file_path>.faiss  ：FAISS 二进制索引
        - <file_path>.history：JSON 格式，存储对话历史

        Parameters
        ----------
        file_path : str
            基础保存路径，str 类型。
        """
        self.vector_store.save(file_path)          # 保存向量库（文档 pickle + FAISS 索引）

        history_path = file_path + ".history"      # 对话历史文件路径，str 类型
        with open(history_path, 'w', encoding='utf-8') as f:
            json.dump(self.conversation_history, f, ensure_ascii=False, indent=2)
            # ensure_ascii=False：保留中文等非 ASCII 字符
            # indent=2：美化 JSON 输出，每层缩进 2 个空格

    def load_system(self, file_path: str):
        """
        从磁盘恢复 RAG 系统完整状态（向量库 + 对话历史）。

        Parameters
        ----------
        file_path : str
            基础文件路径，str 类型，与 save_system() 时保持一致。
        """
        self.vector_store.load(file_path)          # 恢复向量库（文档列表 + FAISS 索引）

        history_path = file_path + ".history"      # 对话历史文件路径，str 类型
        if os.path.exists(history_path):           # 历史文件存在时才加载（首次运行可能不存在）
            with open(history_path, 'r', encoding='utf-8') as f:
                self.conversation_history = json.load(f)
                # 反序列化对话历史，List[Dict[str, str]] 类型

## 四、演示与运行

### 4.1 环境变量与 .env 文件说明

#### python-dotenv 包

`python-dotenv` 是一个轻量级 Python 库，用于将 `.env` 文件中的键值对加载到系统环境变量（`os.environ`）中。

**安装：**
```bash
pip install python-dotenv
```

**核心 API：**

| 函数 | 说明 |
|------|------|
| `load_dotenv()` | 自动查找并加载 `.env` 文件（优先当前目录，向上递归） |
| `os.getenv("KEY")` | 读取环境变量，未设置时返回 `None` |
| `os.getenv("KEY", "默认值")` | 读取环境变量，未设置时返回指定默认值 |

---

#### .env 文件

`.env` 是存放敏感配置的纯文本文件，**每行一个键值对**，格式如下：

```
DEEPSEEK_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
DB_HOST=localhost
DB_PORT=5432
```

**注意事项：**

- `.env` 文件**不要提交到 Git**，需将其加入 `.gitignore`：
  ```
  # .gitignore
  .env
  ```
- 可在项目中提供 `.env.example` 作为模板（不含真实密钥），方便团队协作：
  ```
  # .env.example
  DEEPSEEK_API_KEY=your_api_key_here
  ```
- 变量值无需加引号（`KEY=value` 而非 `KEY="value"`），但带空格时需加引号

### 4.2 load_dotenv() 原理与自定义文件名

#### load_dotenv() 的本质

`load_dotenv()` 本质上是一个**纯文本解析器**：逐行读取文件，按 `KEY=VALUE` 格式解析，将结果写入 `os.environ`。

```
文件内容（任意文件名）      →    os.environ（运行时环境变量）
DEEPSEEK_API_KEY=sk-xxx         os.environ["DEEPSEEK_API_KEY"] = "sk-xxx"
DB_HOST=localhost                os.environ["DB_HOST"] = "localhost"
```

效果等同于在终端中手动执行 `export DEEPSEEK_API_KEY=sk-xxx`，只不过改为从文件读取。

---

#### 文件名对 load_dotenv() 没有限制

`load_dotenv()` 只看**文件内容格式**，不限制文件名或扩展名：

```python
load_dotenv()                              # 默认：自动查找当前及父目录中的 .env
load_dotenv(dotenv_path=".env.production") # 生产环境配置（隐藏文件）
load_dotenv(dotenv_path="config.env")      # xxx.env 格式（普通文件）
load_dotenv(dotenv_path="key.config")      # 任意扩展名，只要内容是 KEY=VALUE
load_dotenv(dotenv_path="secrets.txt")     # 普通文本文件也行
```

| 文件名 | 是否隐藏文件（Linux/Mac） | 是否需要手动指定路径 |
|--------|--------------------------|----------------------|
| `.env` | 是 | 否，自动查找 |
| `.env.production` | 是 | 是 |
| `config.env` | 否 | 是 |
| `key.config` | 否 | 是 |

**"是否需要手动指定路径"的含义：**

- **否（自动查找）**：直接写 `load_dotenv()` 即可，它内置查找逻辑，从当前目录开始逐级向上搜索，直到找到 `.env` 文件为止，无需在代码中写明路径。
- **是（需手动指定）**：`load_dotenv()` 只自动查找名为 `.env` 的文件，对其他文件名一无所知，必须在代码中通过 `dotenv_path` 参数显式告知文件位置：

```python
load_dotenv()                        # ×  找不到 key.config，静默跳过
load_dotenv(dotenv_path="key.config") # ✓  明确告知文件名，正常加载
```

> **结论**：文件名叫什么、扩展名是什么对功能毫无影响，`.env` 只是约定俗成的默认名称，使用它可以省去 `dotenv_path` 参数的书写。

In [32]:
from dotenv import load_dotenv  # 加载 .env 文件中的键值对到系统环境变量，需安装 python-dotenv
import os                       # 标准库，用于通过 os.getenv() 读取环境变量

load_dotenv()
# 自动在当前目录及父目录中查找 .env 文件并加载
# 将文件中的键值对（如 DEEPSEEK_API_KEY=sk-xxx）注入到 os.environ 中

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
# 从环境变量中读取 API 密钥，str 类型
# .env 文件未找到或变量未设置时返回 None，需确保 .env 文件存在且包含该变量

rag = RAGSystem(DEEPSEEK_API_KEY)
# 实例化 RAG 系统，RAGSystem 类型
# 内部自动初始化 DeepSeekClient、DocumentProcessor、VectorStore（含嵌入模型）

# ---- 构建示例知识库（五段文本：前三段 AI 领域科普，后两段与 AI 无关用于测试检索相关性）----
sample_texts = [
    """
    人工智能（AI）是计算机科学的一个分支，致力于创建能够模拟人类智能的系统。
    AI 系统可以学习、推理、解决问题和理解语言。机器学习是 AI 的一个重要子领域，
    它使计算机能够从数据中学习而无需明确编程。深度学习是机器学习的一个子集，
    使用神经网络来模拟人脑的工作方式。
    """,  # 文本 0：AI 基础概念，str 类型
    """
    自然语言处理（NLP）是人工智能的一个重要分支，专注于让计算机理解、
    解释和生成人类语言。NLP 的应用包括机器翻译、情感分析、问答系统和
    文本摘要。现代 NLP 系统大多基于深度学习模型，如 Transformer 架构。
    """,  # 文本 1：NLP 概念介绍，str 类型
    """
    计算机视觉是人工智能的另一个重要领域，致力于让计算机能够识别和理解
    图像和视频内容。常见的计算机视觉任务包括图像分类、目标检测、
    图像分割和人脸识别。卷积神经网络（CNN）是计算机视觉中最常用的模型。
    """,  # 文本 2：计算机视觉概念介绍，str 类型
    """
    光合作用是植物、藻类和某些细菌将光能转化为化学能的过程。
    植物通过叶绿素吸收阳光，利用二氧化碳和水合成葡萄糖，同时释放氧气。
    光合作用分为光反应和暗反应两个阶段，是地球上几乎所有生命能量的最终来源，
    也是大气中氧气的主要来源。
    """,  # 文本 3：生物学——光合作用，str 类型（与 AI 无关）
    """
    板块构造学说认为地球岩石圈由若干大板块组成，这些板块在软流圈上缓慢移动。
    板块之间的相互作用导致了地震、火山喷发和山脉隆起等地质现象。
    著名的板块边界包括环太平洋火山带和喜马拉雅山脉的碰撞边界，
    大陆漂移现象正是板块运动长期积累的结果。
    """,  # 文本 4：地质学——板块构造，str 类型（与 AI 无关）
]
# sample_texts: List[str] 类型，长度 5（示例文本总数）

rag.add_documents_from_text(sample_texts)
# 将五段文本分块、向量化并存入 FAISS 索引，完成知识库构建

# ---- 交互式查询循环 ----
print("=== DeepSeek RAG 系统 ===")
print("输入问题进行查询，输入 'quit' 退出")
print("输入 'clear' 清除对话历史")
print("输入 'save'  保存系统状态")
print("-" * 50)

while True:   # 持续监听用户输入，直到输入 'quit'
    question = input("\n请输入您的问题: ").strip()
    # question: str 类型，.strip() 去除首尾空白字符

    if question.lower() == 'quit':        # 退出指令：终止循环，结束程序
        break

    elif question.lower() == 'clear':     # 清除历史指令：重置对话历史
        rag.clear_history()
        print("对话历史已清除")
        continue

    elif question.lower() == 'save':      # 保存指令：持久化系统状态
        rag.save_system("rag_system.pkl")
        print("系统状态已保存至 rag_system.pkl")
        continue

    elif not question:                    # 空输入：跳过，重新等待
        continue

    # ---- 执行 RAG 查询 ----
    print("正在检索相关文档并生成回答...")
    result = rag.query(question, use_history=True)
    # result: Dict[str, Any] 类型
    # 含键：answer(str)、sources(List[Dict])、context(str)、relevant_documents(int)

    print(f"\n回答:\n{result['answer']}")
    # 打印 LLM 生成的回答文本，str 类型

    print(f"\n本次参考了 {result['relevant_documents']} 个相关文档片段")
    # 打印实际检索到的文档数量，int 类型

    show_sources = input("\n是否显示来源信息? (y/n): ").lower() == 'y'
    # show_sources: bool 类型，True 表示用户希望查看来源详情

    if show_sources:
        print("\n来源信息:")
        for i, source in enumerate(result['sources']):   # 遍历来源文档列表
            print(f"  {i + 1}. 文档 ID  : {source['document_id']}")
            # 打印文档唯一 ID，str 类型
            print(f"     内容预览: {source['content_preview']}")
            # 打印内容前 200 字，str 类型
            print()  # 每条来源信息后空一行，提升可读性

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

使用 sentence-transformers 模型: all-MiniLM-L6-v2
正在向量化 5 个文档...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

成功入库 5 个文档，当前知识库总量: 5 个
=== DeepSeek RAG 系统 ===
输入问题进行查询，输入 'quit' 退出
输入 'clear' 清除对话历史
输入 'save'  保存系统状态
--------------------------------------------------
正在检索相关文档并生成回答...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


回答:
光合作用是植物、藻类和某些细菌将光能转化为化学能的过程。植物通过叶绿素吸收阳光，利用二氧化碳和水合成葡萄糖，同时释放氧气。这一过程分为光反应和暗反应两个阶段，是地球上几乎所有生命能量的最终来源，也是大气中氧气的主要来源。

本次参考了 5 个相关文档片段

来源信息:
  1. 文档 ID  : text_3_0
     内容预览: 光合作用是植物、藻类和某些细菌将光能转化为化学能的过程。
    植物通过叶绿素吸收阳光，利用二氧化碳和水合成葡萄糖，同时释放氧气。
    光合作用分为光反应和暗反应两个阶段，是地球上几乎所有生命能量的最终来源，
    也是大气中氧气的主要来源。

  2. 文档 ID  : text_2_0
     内容预览: 计算机视觉是人工智能的另一个重要领域，致力于让计算机能够识别和理解
    图像和视频内容。常见的计算机视觉任务包括图像分类、目标检测、
    图像分割和人脸识别。卷积神经网络（CNN）是计算机视觉中最常用的模型。

  3. 文档 ID  : text_1_0
     内容预览: 自然语言处理（NLP）是人工智能的一个重要分支，专注于让计算机理解、
    解释和生成人类语言。NLP 的应用包括机器翻译、情感分析、问答系统和
    文本摘要。现代 NLP 系统大多基于深度学习模型，如 Transformer 架构。

  4. 文档 ID  : text_0_0
     内容预览: 人工智能（AI）是计算机科学的一个分支，致力于创建能够模拟人类智能的系统。
    AI 系统可以学习、推理、解决问题和理解语言。机器学习是 AI 的一个重要子领域，
    它使计算机能够从数据中学习而无需明确编程。深度学习是机器学习的一个子集，
    使用神经网络来模拟人脑的工作方式。

  5. 文档 ID  : text_4_0
     内容预览: 板块构造学说认为地球岩石圈由若干大板块组成，这些板块在软流圈上缓慢移动。
    板块之间的相互作用导致了地震、火山喷发和山脉隆起等地质现象。
    著名的板块边界包括环太平洋火山带和喜马拉雅山脉的碰撞边界，
    大陆漂移现象正是板块运动长期积累的结果。

